# Pertemuan 12A — Praproses dan Pembagian Data
## Pekerjaan bersama satu kelompok

Pembelajaran Mesin INF62325 · S1 Informatika · UIN Siber Syekh Nurjati Cirebon

### Siapa yang menjalankan notebook ini

**Satu orang saja**, lalu berkas hasilnya dibagikan ke dua anggota lain. Boleh juga
dijalankan bersama dalam satu folder Drive yang dibagi ke tiga orang. Yang tidak boleh
adalah tiga orang menjalankannya masing-masing di Drive sendiri, karena hasil
pembagian datanya akan berbeda dan ketiga varian menjadi tidak dapat dibandingkan.

### Tujuan

Menyiapkan satu dasar data yang dipakai bertiga: data sudah dibersihkan, sudah dibagi
menjadi latih dan uji, dan sudah punya fungsi praproses yang sama.

### Luaran yang dikumpulkan

| Luaran | Bentuk |
| --- | --- |
| Ukuran data latih dan data uji | tercetak di notebook |
| Jumlah kolom setelah one-hot encoding | tercetak di notebook |
| Kunci data | tercetak di notebook, dicatat di laporan |
| Skor tolok ukur | tercetak di notebook |
| `dasar_bersama.joblib` | berkas di folder proyek pada Drive |

### Prasyarat

Berkas `bank-additional-full.csv` sudah ada di Google Drive Anda. Letaknya bebas;
notebook akan mencarinya sendiri.

---
## Langkah 0 — Samakan versi scikit-learn

Jalankan dua sel berikut setiap kali membuka notebook ini, sebelum sel apa pun
yang lain. Berkas model menyimpan rujukan ke kelas internal scikit-learn versi
tertentu, sehingga versi pelatihan dan versi pemuatan harus sama di seluruh
pertemuan. Angka 1.9.1 berlaku November 2026.

In [29]:
!pip install -q scikit-learn==1.9.1

**Wajib sekarang:** buka menu **Runtime**, pilih **Restart session**, lalu jalankan
ulang dari sel paling atas. Tanpa restart, Python masih memakai versi lama yang
sudah terlanjur dimuat ke memori.

In [30]:
import sklearn, pandas as pd, numpy as np, joblib

print('scikit-learn :', sklearn.__version__)
print('pandas       :', pd.__version__)

assert sklearn.__version__ == '1.9.1', (
    'Versi belum sesuai. Jalankan sel pip di atas, lalu '
    'Runtime -> Restart session, lalu ulangi dari sel pertama.'
)
print('Versi sudah sesuai.')

scikit-learn : 1.9.1
pandas       : 2.2.3
Versi sudah sesuai.


---
## Langkah 1 — Memuat data

Sel berikut memasang Drive, mencari berkas datanya, lalu membacanya. Tidak ada
`PATH` yang perlu diubah. Berkas ini memakai titik koma sebagai pemisah kolom.

In [31]:
import glob, os, time, hashlib
from google.colab import drive

drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/proyek_akhir_ml'
os.makedirs(FOLDER, exist_ok=True)

print('Mencari berkas data di Google Drive, mohon tunggu sebentar...')
temuan = sorted(glob.glob('/content/drive/MyDrive/**/avocado*.csv',
                          recursive=True))

if not temuan:
    raise FileNotFoundError(
        'Berkas avocado.csv tidak ditemukan di Google Drive Anda. '
        'Unggah dulu berkasnya, tunggu sampai tersinkron, lalu ulangi sel ini.'
    )

PATH = temuan[0]
if len(temuan) > 1:
    print(f'Ada {len(temuan)} salinan, yang dipakai salinan pertama:')
    for t in temuan:
        print('  -', t)

mentah = pd.read_csv(PATH, sep=',', index_col=0)
print('\nBerkas dipakai :', PATH)
print('Ukuran data    :', mentah.shape)

if mentah.shape != (18249, 14):
    print('\nPERHATIAN: ukuran seharusnya (18249, 14).')
    print('Kemungkinan berkasnya varian lain, misalnya bank.csv atau bank-full.csv.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mencari berkas data di Google Drive, mohon tunggu sebentar...

Berkas dipakai : /content/drive/MyDrive/proyek_akhir_ml/avocado.csv
Ukuran data    : (18249, 13)

PERHATIAN: ukuran seharusnya (18249, 14).
Kemungkinan berkasnya varian lain, misalnya bank.csv atau bank-full.csv.


### Tiga temuan Pertemuan 11 yang dipakai di sini

1. Kelas target tidak seimbang, hanya 11,3 persen menjawab `yes`. Metrik utama
   karena itu macro-F1, bukan accuracy, dan pembagian data wajib memakai `stratify`.
2. Kolom `pdays` memakai nilai sandi 999 untuk nasabah yang belum pernah dihubungi,
   dan 999 muncul pada sekitar 96 persen baris. Kolom ini diubah menjadi penanda.
3. Kolom `duration` tidak boleh dipakai. Durasi panggilan baru diketahui setelah
   panggilan selesai, sehingga tidak tersedia saat aplikasi memprediksi.

In [32]:

TARGET = 'type'
SEED = 42
N_SUBSAMPEL = 8000

df = mentah.copy()

# Hapus kolom nomor urut
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Ubah tanggal menjadi bulan
df['bulan'] = pd.to_datetime(df['Date']).dt.month
df = df.drop(columns=['Date'])

# Hapus wilayah agregat
wilayah_agregat = [
    'California',
    'GreatLakes',
    'Midsouth',
    'Northeast',
    'Plains',
    'SouthCentral',
    'Southeast',
    'West',
    'TotalUS'
]

df = df[~df['region'].isin(wilayah_agregat)].copy()

# Hapus kolom total yang merupakan penjumlahan komponen
df = df.drop(columns=['Total Volume', 'Total Bags'])

print('Setelah pembersihan:', df.shape)
print('\nDistribusi kelas:')
print(df[TARGET].value_counts())
print('\nProporsi kelas:')
print(df[TARGET].value_counts(normalize=True).round(4))


Setelah pembersihan: (15207, 11)

Distribusi kelas:
type
conventional    7605
organic         7602
Name: count, dtype: int64

Proporsi kelas:
type
conventional    0.5001
organic         0.4999
Name: proportion, dtype: float64


### Mengapa diambil subsampel

k-NN tidak melatih apa pun; ia menyimpan seluruh data latih lalu menghitung jarak ke
semuanya pada setiap prediksi. Penyetelan parameter di Varian 2 mencoba 64 kombinasi
pada 5 lipatan, jadi 320 kali pelatihan. Pada 41.188 baris itu memakan lebih dari 20
menit, sedangkan pada subsampel 8.000 baris selesai sekitar 35 detik.

Subsampel diambil berstrata dengan `random_state` tetap, sehingga proporsi kelas
terjaga dan siapa pun yang menjalankan notebook ini memperoleh baris yang sama persis.
Keputusan ini wajib ditulis di bab Metodologi laporan, lengkap dengan alasannya.

---
## Langkah 2 — Membagi data sekali untuk tiga varian

Inilah sel yang paling menentukan keabsahan proyek kelompok Anda. Pembagian dibuat
**satu kali di sini**, dan ketiga varian nanti memuatnya, bukan membuatnya ulang.

Sel ini juga menghitung **kunci data**, yaitu sidik ringkas dari daftar baris yang
masuk ke data latih dan data uji. Catat kunci itu. Pertemuan 13 akan menolak
menggabungkan varian yang kuncinya berbeda.

In [33]:
from sklearn.model_selection import train_test_split

df, _ = train_test_split(df, train_size=N_SUBSAMPEL,
                         stratify=df[TARGET], random_state=SEED)
df = df.reset_index(drop=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]

numerik = X.select_dtypes(include='number').columns.tolist()
kategorik = X.select_dtypes(exclude='number').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

teks_kunci = (','.join(map(str, X_train.index)) + '|' +
              ','.join(map(str, X_test.index)))
KUNCI_DATA = hashlib.md5(teks_kunci.encode()).hexdigest()[:12]

print('Subsampel      :', df.shape)
print('Fitur numerik  :', len(numerik), 'kolom')
print('Fitur kategorik:', len(kategorik), 'kolom')
print('Latih          :', X_train.shape)
print('Uji            :', X_test.shape)
print('\nDistribusi kelas data latih:')
print(y_train.value_counts())
print('\nDistribusi kelas data uji:')
print(y_test.value_counts())
print('Proporsi conventional di latih:',round((y_train == 'conventional').mean(), 4))
print('Proporsi conventional di uji:',round((y_test == 'conventional').mean(), 4))
print('\nKUNCI DATA     :', KUNCI_DATA)
print('Catat kunci ini. Ketiga anggota harus memperoleh kunci yang sama.')

Subsampel      : (8000, 11)
Fitur numerik  : 9 kolom
Fitur kategorik: 1 kolom
Latih          : (6400, 10)
Uji            : (1600, 10)

Distribusi kelas data latih:
type
conventional    3201
organic         3199
Name: count, dtype: int64

Distribusi kelas data uji:
type
organic         800
conventional    800
Name: count, dtype: int64
Proporsi conventional di latih: 0.5002
Proporsi conventional di uji: 0.5

KUNCI DATA     : a0a218fc5df0
Catat kunci ini. Ketiga anggota harus memperoleh kunci yang sama.


---
## Langkah 3 — Fungsi praproses yang dipakai bertiga

Satu aturan menentukan keabsahan seluruh proyek: data dibagi lebih dahulu, lalu semua
transformasi dipasang hanya pada data latih. Pelaksanaannya memakai `ColumnTransformer`
yang dibungkus di dalam `Pipeline`, sehingga imputasi dan penskalaan ikut terpasang
ulang pada setiap lipatan validasi silang.

Perhatikan parameter `pakai_skala`. k-NN bekerja atas dasar jarak sehingga menuntut
penskalaan; Naive Bayes menghitung probabilitas tiap fitur secara terpisah sehingga
tidak. Perbedaan ini disengaja, dan Varian 1 akan membuktikannya dengan angka.

Fungsi yang sama persis ditulis ulang di ketiga notebook varian. Jangan mengubahnya
hanya di salah satu notebook; kalau diubah, ubah di ketiganya.

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def buat_preprocessor(kolom_numerik, kolom_kategorik, pakai_skala):
    # pakai_skala=True untuk model berbasis jarak seperti k-NN.
    # pakai_skala=False untuk Naive Bayes yang tidak memakai jarak.
    langkah_numerik = [('imputasi', SimpleImputer(strategy='median'))]
    if pakai_skala:
        langkah_numerik.append(('skala', StandardScaler()))

    alur_kategorik = Pipeline([
        ('imputasi', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    return ColumnTransformer([
        ('num', Pipeline(langkah_numerik), kolom_numerik),
        ('kat', alur_kategorik, kolom_kategorik),
    ])


print('Fungsi buat_preprocessor siap dipakai.')

Fungsi buat_preprocessor siap dipakai.


In [35]:
contoh = buat_preprocessor(numerik, kategorik, pakai_skala=True).fit_transform(X_train)
print('Jumlah kolom setelah encoding:', contoh.shape[1])

Jumlah kolom setelah encoding: 54


---
## Langkah 4 — Tolok ukur yang harus dikalahkan

Sebelum melatih model apa pun, kelompok perlu tahu angka terendah yang pantas disebut
hasil. `DummyClassifier` dengan `strategy='most_frequent'` selalu menjawab `no` tanpa
melihat data sama sekali. Accuracy-nya tinggi karena kelasnya tidak seimbang, tetapi
macro-F1-nya rendah. Perbandingan kedua angka itu menjelaskan sendiri mengapa metrik
utama proyek ini macro-F1.

In [36]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

tolok = DummyClassifier(strategy='most_frequent')
skor_tolok = cross_val_score(tolok, X_train, y_train, cv=cv, scoring='f1_macro')
skor_tolok_acc = cross_val_score(tolok, X_train, y_train, cv=cv, scoring='accuracy')

print('Baseline klasifikasi Avocado:')
print('  f1_macro :', round(float(skor_tolok.mean()), 4))
print('  accuracy :', round(float(skor_tolok_acc.mean()), 4))
print('\nKetiga varian wajib mengalahkan angka f1_macro di atas.')
print('Varian yang accuracy-nya tinggi tetapi f1_macro-nya tidak jauh dari angka ini')
print('berarti hampir tidak pernah berhasil menemukan kelas yes.')

Baseline klasifikasi Avocado:
  f1_macro : 0.3334
  accuracy : 0.5002

Ketiga varian wajib mengalahkan angka f1_macro di atas.
Varian yang accuracy-nya tinggi tetapi f1_macro-nya tidak jauh dari angka ini
berarti hampir tidak pernah berhasil menemukan kelas yes.


---
## Langkah 5 — Menyimpan dasar bersama

In [37]:
dasar = {
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'numerik': numerik, 'kategorik': kategorik,
    'seed': SEED, 'kunci_data': KUNCI_DATA,
    'kolom_encoding': int(contoh.shape[1]),
    'f1_tolok_ukur': round(float(skor_tolok.mean()), 4),
    'akurasi_tolok_ukur': round(float(skor_tolok_acc.mean()), 4),
    'versi_sklearn': sklearn.__version__,
}

jalur = os.path.join(FOLDER, 'dasar_bersama.joblib')
joblib.dump(dasar, jalur)
print('Tersimpan:', jalur)
print('Ukuran   :', round(os.path.getsize(jalur) / 1024, 1), 'KB')
print('\nBagikan berkas ini ke dua anggota lain, atau pastikan ketiganya memakai')
print('folder Drive yang sama.')

Tersimpan: /content/drive/MyDrive/proyek_akhir_ml/dasar_bersama.joblib
Ukuran   : 770.3 KB

Bagikan berkas ini ke dua anggota lain, atau pastikan ketiganya memakai
folder Drive yang sama.


### Pembagian tugas tiga varian

Satu kelompok tiga orang, satu metode, tiga varian. Ketiganya berjalan di atas
pembagian data yang sama, sehingga hasilnya dapat disusun menjadi satu tabel
perbandingan.

| Varian | Pengerja | Isi pekerjaan | Notebook |
| --- | --- | --- | --- |
| Varian 1 | anggota 1 | k-NN dasar, k=5 tanpa penyetelan, sekaligus bukti bahwa penskalaan wajib | `P12B_Varian1_kNN_Dasar.ipynb` |
| Varian 2 | anggota 2 | penyetelan parameter k-NN: jumlah tetangga, pembobotan, metrik jarak | `P12C_Varian2_Penyetelan_kNN.ipynb` |
| Varian 3 | anggota 3 | Naive Bayes sebagai algoritma pembanding | `P12D_Varian3_Naive_Bayes.ipynb` |

Dua pekerjaan berikut adalah **keluaran bersama kelompok**, bukan milik satu anggota:

1. Tabel perbandingan ketiga varian, dikerjakan di Pertemuan 13.
2. Satu eksperimen pemilihan fitur, dilaporkan sebagai tabel tersendiri, juga di
   Pertemuan 13.

---
## Daftar periksa sebelum notebook ini ditutup

- [x] Ukuran data mentah tercetak `(18249, 14)`
- [x] Data latih 6.400 baris dan data uji 1.600 baris
- [x] Jumlah kolom setelah encoding tercetak 54
- [x] Kunci data tercetak dan sudah dicatat
- [x] Skor tolok ukur tercetak
- [x] Berkas `dasar_bersama.joblib` muncul di folder proyek pada Drive
- [x] Ketiga anggota sudah tahu varian mana yang menjadi tanggung jawabnya